# Lesson 8A: Recommender Systems Theory

## Introduction

Every day you see recommendations:
- Netflix suggests movies you might enjoy
- Amazon recommends products based on your browsing
- Spotify creates personalized playlists
- YouTube suggests videos to watch next

These aren't random suggestions - they're powered by **recommender systems**, one of the most commercially successful applications of machine learning.

Think about it: You've rated 50 movies on Netflix. There are 10,000 movies in the catalog. How does Netflix know which of the remaining 9,950 movies you'll love?

The insight: People who rated movies similarly to you in the past probably have similar tastes. If users with similar preferences loved a movie you haven't seen, you'll probably love it too.

In this lesson, we'll:
1. Understand the two main approaches: Collaborative vs. Content-based filtering
2. Derive the collaborative filtering algorithm from first principles
3. Learn matrix factorization for efficient recommendations
4. Implement gradient descent for learning user and item features
5. Handle the cold start problem
6. Build a movie recommendation system from scratch

Then in Lesson 8B:
1. Use production libraries (Surprise, Implicit)
2. Handle implicit feedback (clicks, views)
3. Implement deep learning approaches with TensorFlow
4. Evaluate recommendation quality

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [Types of Recommender Systems](#types-of-recommender-systems)
4. [Collaborative Filtering](#collaborative-filtering)
5. [Matrix Factorization](#matrix-factorization)
6. [Cost Function](#cost-function)
7. [Gradient Descent for Recommendations](#gradient-descent)
8. [From-Scratch Implementation](#from-scratch-implementation)
9. [MovieLens Dataset](#movielens-dataset)
10. [Making Predictions](#making-predictions)
11. [Finding Similar Items](#finding-similar-items)
12. [Handling Bias](#handling-bias)
13. [Cold Start Problem](#cold-start-problem)
14. [Conclusion](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

| Library | Purpose |
|---------|--------|
| NumPy | Numerical computing |
| Pandas | Data manipulation |
| Matplotlib/Seaborn | Visualization |
| Scikit-learn | Metrics |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple
from numpy.typing import NDArray

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Libraries loaded!")

<a name="types-of-recommender-systems"></a>
## Types of Recommender Systems

### 1. Content-Based Filtering
**Idea**: Recommend items similar to what the user liked before

- Analyzes item features (genre, director, actors for movies)
- Builds user profile from items they liked
- Recommends items matching the profile

**Pros**: No need for other users' data, can recommend niche items
**Cons**: Limited to features we can extract, can't discover new interests

### 2. Collaborative Filtering
**Idea**: Recommend what similar users liked

- Uses only ratings, no item features needed
- Finds users with similar tastes
- Recommends items those similar users enjoyed

**Pros**: Discovers new interests, no feature engineering needed
**Cons**: Cold start problem, popularity bias

### 3. Hybrid
Combines both approaches for best results (Netflix, Amazon)

<a name="collaborative-filtering"></a>
## Collaborative Filtering

### The Problem Setup

**Given**:
- $n_u$ users
- $n_m$ movies (items)
- Rating matrix $R$ where $R_{ij}$ is user $i$'s rating of movie $j$
- Most entries are missing (users haven't rated most movies)

**Goal**: Predict missing ratings

### Example:

| User | Titanic | Avatar | Star Wars | Inception | ? |
|------|---------|--------|-----------|-----------|---|
| Alice| 5 | ? | 1 | ? | |
| Bob  | ? | 4 | ? | 3 | |
| Carol| 5 | ? | 2 | ? | |
| Dave | ? | 5 | ? | 4 | |

**Observation**: Alice and Carol have similar taste (both loved Titanic, disliked Star Wars)

**Prediction**: If Carol loved a movie Alice hasn't seen, Alice will probably love it too!

<a name="matrix-factorization"></a>
## Matrix Factorization

### The Key Insight

The rating matrix can be approximated as the product of two lower-rank matrices:

$$R \approx X \Theta^T$$

Where:
- $X$ is $(n_m \times k)$ - movie features
- $\Theta$ is $(n_u \times k)$ - user preferences  
- $k$ is the number of latent features (much smaller than $n_m$ or $n_u$)

**Predicted rating**: $\hat{r}_{ij} = x_i^T \theta_j$

### What are these features?

For movies, they might capture:
- Romance level
- Action level  
- Comedy level
- How recent/classic
- Budget/production value

The algorithm **learns these automatically** from the ratings!

<a name="cost-function"></a>
## Cost Function

We want predicted ratings to match actual ratings:

### Basic Cost Function:

$$J(X, \Theta) = \frac{1}{2} \sum_{(i,j):r_{ij}=1} (x_i^T \theta_j - y_{ij})^2$$

Where:
- Sum is only over rated movies
- $r_{ij} = 1$ if user $j$ rated movie $i$
- $y_{ij}$ is the actual rating

### With Regularization:

$$J(X, \Theta) = \frac{1}{2} \sum_{(i,j):r_{ij}=1} (x_i^T \theta_j - y_{ij})^2 + \frac{\lambda}{2} \sum_{i=1}^{n_m} \|x_i\|^2 + \frac{\lambda}{2} \sum_{j=1}^{n_u} \|\theta_j\|^2$$

Regularization prevents overfitting to the limited ratings we have.

<a name="gradient-descent"></a>
## Gradient Descent for Recommendations

### Update Rules:

For each movie feature vector $x_i$:
$$x_i := x_i - \alpha \frac{\partial J}{\partial x_i}$$

For each user preference vector $\theta_j$:
$$\theta_j := \theta_j - \alpha \frac{\partial J}{\partial \theta_j}$$

### Gradients:

$$\frac{\partial J}{\partial x_i} = \sum_{j:r_{ij}=1} (x_i^T \theta_j - y_{ij}) \theta_j + \lambda x_i$$

$$\frac{\partial J}{\partial \theta_j} = \sum_{i:r_{ij}=1} (x_i^T \theta_j - y_{ij}) x_i + \lambda \theta_j$$

### Algorithm:
1. Initialize $X$ and $\Theta$ randomly (small values)
2. For each iteration:
   - Compute gradients for all $x_i$ and $\theta_j$
   - Update all parameters simultaneously
3. Repeat until convergence

<a name="from-scratch-implementation"></a>
## From-Scratch Implementation

In [ ]:
class CollaborativeFiltering:
    """Collaborative Filtering with Matrix Factorization."""
    
    def __init__(self, n_features=10, learning_rate=0.01, reg_lambda=0.1, n_iterations=1000):
        self.k = n_features
        self.lr = learning_rate
        self.lam = reg_lambda
        self.n_iters = n_iterations
        self.X = None  # Movie features
        self.Theta = None  # User preferences
        self.costs = []
    
    def fit(self, R: NDArray, Y: NDArray):
        """
        Train the collaborative filtering model.
        
        Args:
            R: (n_movies, n_users) binary matrix, 1 if rated, 0 otherwise
            Y: (n_movies, n_users) ratings matrix
        """
        n_movies, n_users = R.shape
        
        # Initialize parameters randomly
        self.X = np.random.randn(n_movies, self.k) * 0.01
        self.Theta = np.random.randn(n_users, self.k) * 0.01
        
        # Gradient descent
        for iteration in range(self.n_iters):
            # Predictions
            predictions = self.X @ self.Theta.T
            
            # Errors (only for rated movies)
            errors = (predictions - Y) * R
            
            # Compute gradients
            X_grad = errors @ self.Theta + self.lam * self.X
            Theta_grad = errors.T @ self.X + self.lam * self.Theta
            
            # Update parameters
            self.X -= self.lr * X_grad
            self.Theta -= self.lr * Theta_grad
            
            # Track cost
            if iteration % 100 == 0:
                cost = self._compute_cost(R, Y)
                self.costs.append(cost)
                if iteration % 200 == 0:
                    print(f"Iteration {iteration}: Cost = {cost:.4f}")
    
    def _compute_cost(self, R: NDArray, Y: NDArray) -> float:
        """Compute the cost function."""
        predictions = self.X @ self.Theta.T
        errors = (predictions - Y) * R
        
        # MSE on rated items + regularization
        cost = 0.5 * np.sum(errors ** 2)
        cost += 0.5 * self.lam * (np.sum(self.X ** 2) + np.sum(self.Theta ** 2))
        return cost
    
    def predict(self, movie_idx: int, user_idx: int) -> float:
        """Predict rating for a specific movie-user pair."""
        return self.X[movie_idx] @ self.Theta[user_idx]
    
    def recommend(self, user_idx: int, n_recommendations=5) -> NDArray:
        """Get top N movie recommendations for a user."""
        predictions = self.X @ self.Theta[user_idx]
        top_indices = np.argsort(predictions)[::-1][:n_recommendations]
        return top_indices
    
    def find_similar_movies(self, movie_idx: int, n_similar=5) -> NDArray:
        """Find movies similar to a given movie."""
        # Compute cosine similarity
        movie_vec = self.X[movie_idx]
        similarities = self.X @ movie_vec / (np.linalg.norm(self.X, axis=1) * np.linalg.norm(movie_vec))
        similar_indices = np.argsort(similarities)[::-1][1:n_similar+1]  # Exclude the movie itself
        return similar_indices

print("✅ Collaborative filtering implementation complete!")

<a name="movielens-dataset"></a>
## MovieLens Dataset

We'll use the MovieLens 100K dataset:
- 100,000 ratings
- 943 users
- 1,682 movies
- Ratings from 1-5

**Note**: In a real implementation, you would download this dataset. For this notebook, we'll create synthetic data that mimics the structure.

In [ ]:
# Create synthetic movie ratings data
n_users = 50
n_movies = 100
sparsity = 0.1  # 10% of movies rated

# Binary matrix: 1 if rated, 0 otherwise
R = (np.random.rand(n_movies, n_users) < sparsity).astype(int)

# Ratings matrix (1-5 stars)
Y = np.random.randint(1, 6, size=(n_movies, n_users)) * R

print(f"Dataset shape: {n_movies} movies × {n_users} users")
print(f"Total possible ratings: {n_movies * n_users:,}")
print(f"Actual ratings: {R.sum():,} ({R.sum()/(n_movies*n_users)*100:.1f}%)")
print(f"\nRating distribution:")
for rating in range(1, 6):
    count = np.sum(Y == rating)
    print(f"  {rating} stars: {count} ({count/R.sum()*100:.1f}%)")

In [ ]:
# Visualize the rating matrix
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(R, aspect='auto', cmap='binary')
plt.title('Which Movies Are Rated (Black = Rated)')
plt.xlabel('Users')
plt.ylabel('Movies')
plt.colorbar(label='Rated')

plt.subplot(1, 2, 2)
plt.imshow(Y, aspect='auto', cmap='YlOrRd', vmin=0, vmax=5)
plt.title('Rating Values')
plt.xlabel('Users')
plt.ylabel('Movies')
plt.colorbar(label='Rating (1-5)')

plt.tight_layout()
plt.show()

<a name="making-predictions"></a>
## Making Predictions

In [ ]:
# Train the model
model = CollaborativeFiltering(n_features=10, learning_rate=0.01, reg_lambda=0.1, n_iterations=1000)
model.fit(R, Y)

# Plot learning curve
plt.figure(figsize=(10, 5))
plt.plot(range(0, len(model.costs)*100, 100), model.costs, linewidth=2)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Cost', fontsize=12)
plt.title('Learning Curve: Collaborative Filtering', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

print("\n✅ Model trained successfully!")

In [ ]:
# Make recommendations for a user
user_idx = 0
recommendations = model.recommend(user_idx, n_recommendations=10)

print(f"Top 10 movie recommendations for User {user_idx}:")
for i, movie_idx in enumerate(recommendations, 1):
    predicted_rating = model.predict(movie_idx, user_idx)
    print(f"{i}. Movie {movie_idx}: Predicted rating = {predicted_rating:.2f}")

<a name="finding-similar-items"></a>
## Finding Similar Items

Once we've learned the movie features, we can find similar movies using cosine similarity:

$$\text{similarity}(i, j) = \frac{x_i^T x_j}{\|x_i\| \|x_j\|}$$

In [ ]:
# Find similar movies
movie_idx = 5
similar_movies = model.find_similar_movies(movie_idx, n_similar=5)

print(f"Movies similar to Movie {movie_idx}:")
for i, similar_idx in enumerate(similar_movies, 1):
    # Compute similarity
    vec1 = model.X[movie_idx]
    vec2 = model.X[similar_idx]
    similarity = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    print(f"{i}. Movie {similar_idx}: Similarity = {similarity:.3f}")

<a name="handling-bias"></a>
## Handling Bias

**Problem**: Some users rate everything high, others rate everything low. Some movies are universally loved or hated.

**Solution**: Mean normalization

$$\hat{r}_{ij} = x_i^T \theta_j + \mu_i$$

Where $\mu_i$ is the mean rating for movie $i$.

This allows us to learn deviations from the mean rather than absolute ratings.

<a name="cold-start-problem"></a>
## Cold Start Problem

### The Problem
- **New users**: No ratings yet → can't find similar users
- **New items**: No one has rated them → can't recommend

### Solutions

1. **For new users**:
   - Ask for a few ratings upfront
   - Use demographic information
   - Recommend popular items initially

2. **For new items**:
   - Use content-based features
   - Cold start promotion (show to sample of users)
   - Hybrid approach

3. **Hybrid systems**:
   - Combine collaborative and content-based
   - Use content features when few ratings available
   - Switch to collaborative as more ratings come in

<a name="conclusion"></a>
## Conclusion

### What We Learned

1. **Collaborative filtering** uses ratings from similar users to make recommendations
2. **Matrix factorization** learns latent features for users and items
3. **Gradient descent** optimizes the feature vectors to minimize prediction error
4. **Mean normalization** handles bias in user ratings
5. **Cold start** is challenging but solvable with hybrid approaches

### Advantages
- No feature engineering needed
- Can discover unexpected patterns
- Works well with implicit feedback

### Limitations
- Requires substantial rating data
- Cold start for new users/items
- Popularity bias
- Difficult to explain recommendations

### When to Use
- ✅ You have user-item interaction data (ratings, clicks, purchases)
- ✅ You want to discover non-obvious patterns
- ✅ Content features are hard to extract
- ❌ You have mostly new users/items (use content-based or hybrid)
- ❌ You need explainable recommendations (use content-based)

### Next: Lesson 8B
Production implementations with Surprise, Implicit, and deep learning approaches!